# 实验 6 — incremental 物化：SQL 有什么变化？

**目标**：把 `fct_daily_rates` 从 `table` 改成 `incremental`，对比 dbt 生成的 SQL 有何不同。

**关键概念**：incremental 模型在首次运行时是全量，之后只处理新数据——节省算力，但需要定义 '新' 是什么。

In [ ]:
import subprocess, os
from pathlib import Path

def run_dbt(cmd):
    result = subprocess.run(
        ["uv", "run", "dbt"] + cmd,
        capture_output=True, text=True, cwd="../dbt_project",
        env={**os.environ, "DBT_PROFILES_DIR": "."}
    )
    return result.stdout + result.stderr

# 先看 table 模式生成的 SQL
out = run_dbt(["compile", "--select", "fct_daily_rates"])
print(out[-1000:])

In [ ]:
# 查看 table 模式编译后的 SQL 文件
compiled = Path("../dbt_project/target/compiled/dlt_dbt_sandbox/models/marts/fct_daily_rates.sql")
if compiled.exists():
    print("=== TABLE 模式编译 SQL ===")
    print(compiled.read_text())

## 步骤：修改为 incremental

把 `dbt_project/models/marts/fct_daily_rates.sql` 改成下面的内容（加了 config 和 is_incremental filter）：

```sql
{{ config(materialized='incremental', unique_key=['rate_date', 'currency']) }}

with stg as (
    select * from {{ ref('stg_ecb_rates') }}
    {% if is_incremental() %}
    where rate_date > (select max(rate_date) from {{ this }})
    {% endif %}
),
...
```

改完后运行下面的 cell。

In [ ]:
# 改完 fct_daily_rates.sql 后再 compile，看 SQL 变化
out = run_dbt(["compile", "--select", "fct_daily_rates", "--no-partial-parse"])
compiled2 = Path("../dbt_project/target/compiled/dlt_dbt_sandbox/models/marts/fct_daily_rates.sql")
if compiled2.exists():
    print("=== INCREMENTAL 模式编译 SQL ===")
    print(compiled2.read_text())

## 思考

- incremental 模式第一次运行（全量）和后续运行（增量）的 SQL 有什么区别？
- `is_incremental()` 宏在什么条件下返回 True？
- 注意：incremental 里用 `LAG()` 窗口函数有个陷阱——增量 batch 里第一行看不到上一个 batch 的最后一行，`rate_change_pct` 会是 NULL。
- 实验结束后，把 `fct_daily_rates.sql` 还原为 `table` 模式，运行 `make build` 确认恢复。